In [ ]:
!pip install pandas_datareader -q

import pandas_datareader.data as web
import pandas as pd
import warnings
import os
from datetime import datetime, timedelta, date

# Suppress warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

# ==========================================
# 1. SETTINGS
# ==========================================
LOOKBACK_DAYS = 80
TOP_PCT       = 25
BTM_PCT       = 25
MAX_RESULTS   = 25    

# SPECIAL MAPPING
# Stooq is quirky. We map standard tickers to Stooq format if needed.
TICKER_MAP = {
    "SPX": "^SPX", "XSP": "^XSP", "DJI": "^DJI", "IXIC": "^ICIC", 
    "BRK.B": "BRK.B", "BRK-B": "BRK.B"
}

RAW_LIST = """
A
AA
AAL
AAP
AAPL
ABBV
ABNB
ABT
ACN
ADBE
ADI
ADM
ADP
ADSK
AEP
AES
AFL
AFRM
AGNC
AI
AIG
AKAM
ALB
ALK
ALL
AMAT
AMD
AMGN
AMRN
AMZN
APA
APH
APTV
AVGO
AXP
BA
BABA
BAC
BAX
BBY
BEN
BIDU
BIIB
BITO
BK
BKR
BMY
BP
BSX
BX
BYND
C
CAH
CAT
CB
CCI
CCJ
CCL
CF
CFG
CHWY
CI
CL
CLF
CLX
CMCSA
CME
CNC
CNP
COF
COIN
COP
COST
CPB
CPRI
CRM
CRON
CRWD
CSCO
CSX
CTVA
CVNA
CVS
CVX
CZR
D
DAL
DD
DE
DELL
DHI
DHR
DIA
DIS
DKNG
DLR
DOCU
DOW
DRI
DVN
DXC
EA
EBAY
ED
EEM
EFA
EIX
EL
EMR
ENPH
EOG
EQR
EQT
ETSY
EVRG
EW
EWJ
EWW
EWY
EWZ
EXC
EXPE
F
FANG
FAST
FCX
FDX
FE
FI
FITB
FOXA
FSLR
FTI
FTV
FXE
FXI
GD
GDX
GE
GILD
GLD
GLW
GM
GME
GOOG
GOOGL
GPRO
GPS
GS
HAL
HBAN
HBI
HCA
HD
HIG
HLT
HOG
HOLX
HON
HPE
HPQ
HYG
IBB
IBIT
IBM
ICE
IEF
INTC
IP
IPG
IRM
IVZ
IWM
IYR
JCI
JD
JETS
JNJ
JNPR
JPM
K
KHC
KMI
KO
KR
KRE
KSS
KWEB
LEN
LI
LKQ
LLY
LNC
LOW
LQD
LUMN
LUV
LVS
LYB
LYFT
MA
MAR
MARA
MAS
MCD
MCHP
MDLZ
MDT
MET
META
MGM
MMM
MO
MOS
MPC
MRK
MRNA
MRVL
MS
MSFT
MSTR
MTB
MTCH
MU
NCLH
NEE
NEM
NET
NFLX
NKE
NOW
NRG
NTAP
NTES
NVAX
NVDA
NWL
NWSA
O
OIH
OKE
OMC
ON
ORCL
OXY
PARA
PBR
PDD
PENN
PEP
PFE
PFG
PG
PGR
PINS
PLTR
PNC
PPL
PRU
PSX
PTON
PYPL
QCOM
QQQ
RBLX
RCL
RF
RIG
RIOT
RIVN
ROKU
ROST
RTX
RUN
SBUX
SCHD
SCHW
SEDG
SHOP
SIRI
SLB
SLV
SMCI
SMH
SNAP
SNOW
SO
SOFI
SOXL
SOXS
SPG
SPX
SPXL
SPXS
SPY
SQQQ
STX
SWKS
SYF
SYY
T
TAP
TBT
TCOM
TDOC
TFC
TGT
TJX
TLT
TMO
TMUS
TPR
TQQQ
TRIP
TRV
TSLA
TSM
TSN
TT
TTD
TTWO
TXN
U
UA
UAA
UAL
UBER
ULTA
UNG
UNH
UNP
UPS
UPST
URBN
USB
USO
UVXY
V
VALE
VFC
VGK
VTR
VXX
VZ
WAB
WBA
WBD
WDC
WFC
WM
WMB
WMT
WU
WY
WYNN
X
XBI
XEL
XHB
XLB
XLC
XLE
XLF
XLI
XLK
XLP
XLRE
XLU
XLV
XLY
XOM
XOP
XRT
XRX
XSP
XYZ
YELP
YINN
ZION
ZM
"""

# ==========================================
# 2. DATA PROCESSING
# ==========================================
def get_clean_tickers():
    raw_tickers = [t.strip() for t in RAW_LIST.split('\n') if t.strip()]
    raw_tickers = list(set(raw_tickers))
    return raw_tickers

def get_stooq_data(tickers):
    """
    Downloads from STOOQ adding .US suffix for correct US data.
    """
    print(f"   [STOOQ] Downloading {len(tickers)} symbols (adding .US suffix)...")
    
    # We need ~100 days of data to calculate the 80-day lookback safely
    start_date = datetime.now() - timedelta(days=150)
    
    # Chunking to prevent URL length errors
    chunk_size = 50
    all_closes = pd.DataFrame()
    
    for i in range(0, len(tickers), chunk_size):
        chunk = tickers[i:i+chunk_size]
        
        # PREPARE TICKERS FOR STOOQ
        # 1. Replace dots with dashes won't work for Stooq, Stooq likes dots.
        # 2. Append .US to standard stocks
        # 3. Indices (starting with ^) usually don't get .US
        stooq_tickers = []
        mapping = {} # To map "AAPL.US" back to "AAPL"
        
        for t in chunk:
            # Map known indices first
            if t in TICKER_MAP:
                stooq_t = TICKER_MAP[t]
                stooq_tickers.append(stooq_t)
                mapping[stooq_t] = t
                continue
            
            # Handle standard stocks
            if "^" not in t:
                # Stooq format: AAPL.US
                stooq_t = f"{t.replace('-', '.')}.US"
                stooq_tickers.append(stooq_t)
                mapping[stooq_t] = t
            else:
                stooq_tickers.append(t)
                mapping[t] = t

        try:
            # Stooq returns data with newest date first (Desc)
            df = web.DataReader(stooq_tickers, 'stooq', start=start_date)
            
            # Extract Closes
            if 'Close' in df:
                closes = df['Close']
            else:
                # Fallback if structure varies
                closes = df
            
            # Rename columns from "AAPL.US" back to "AAPL"
            closes = closes.rename(columns=mapping)
            
            # Sort index to be Oldest -> Newest (Ascending)
            closes = closes.sort_index()
            
            if all_closes.empty:
                all_closes = closes
            else:
                all_closes = pd.concat([all_closes, closes], axis=1)
                
            print(f"      -> Chunk {i//chunk_size + 1} complete.")
            
        except Exception as e:
            print(f"      -> Chunk failed: {e}")
            
    return all_closes

# ==========================================
# 3. MAIN SCANNER
# ==========================================
def run_scanner():
    print("--- STEP 1: PROCESSING TICKER LIST ---")
    yahoo_tickers = get_clean_tickers()
    print(f"   Loaded {len(yahoo_tickers)} unique symbols.")
    
    print("\n--- STEP 2: MARKET DATA (STOOQ) ---")
    data = get_stooq_data(yahoo_tickers)
    
    print("\n--- STEP 3: ANALYZING ZONES ---")
    
    # Ensure date index is sorted Ascending (Past -> Today)
    data = data.sort_index()
    
    # Slice the last 80 days
    recent_data = data.tail(LOOKBACK_DAYS)
    
    premium_list = []
    discount_list = []
    
    for col in recent_data.columns:
        try:
            series = recent_data[col].dropna()
            # Need at least 20 days of data to be meaningful
            if len(series) < 20: continue
            
            curr = series.iloc[-1]
            hi   = series.max()
            lo   = series.min()
            rng  = hi - lo
            
            if rng == 0: continue
            
            loc = (curr - lo) / rng
            
            item = {'ticker': col, 'loc': round(loc*100, 1), 'price': round(curr, 2)}
            
            if loc >= (1.0 - (TOP_PCT/100)):
                premium_list.append(item)
            elif loc <= (BTM_PCT/100):
                discount_list.append(item)
                
        except Exception:
            continue

    # ==========================================
    # FILTERING (KEEP ONLY TOP 25)
    # ==========================================
    
    discount_list.sort(key=lambda x: x['loc'])
    final_discount = discount_list[:MAX_RESULTS]
    
    premium_list.sort(key=lambda x: x['loc'], reverse=True)
    final_premium = premium_list[:MAX_RESULTS]

    # ==========================================
    # OUTPUT
    # ==========================================
    print("\n" + "="*60)
    print(f"MOST DISCOUNTED (Showing Top {len(final_discount)})")
    print("="*60)
    for s in final_discount: 
        print(f"{s['ticker']:<8} {s['loc']:<6}% ${s['price']}")

    print("\n" + "="*60)
    print(f"MOST PREMIUM (Showing Top {len(final_premium)})")
    print("="*60)
    for s in final_premium:
        print(f"{s['ticker']:<8} {s['loc']:<6}% ${s['price']}")

    # ==========================================
    # TRADINGVIEW IMPORT STRINGS
    # ==========================================
    print("\n\n" + "#"*60)
    print("TRADINGVIEW IMPORT STRINGS")
    print("#"*60)
    
    print(f"\n>>> DISCOUNT LIST:")
    print(", ".join([x['ticker'] for x in final_discount]))
    
    print(f"\n>>> PREMIUM LIST:")
    print(", ".join([x['ticker'] for x in final_premium]))

run_scanner()